# dashboard_manifest_readiness_gate_v1

验证 dashboard view model 和静态 HTML 是否把 dashboard export 的 `manifest_source` 作为 readiness gate 展示。目标是让 review 页面能直接看到 paper/API 为什么允许或阻断执行。

In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

EXP = "dashboard_manifest_readiness_gate_v1"
ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dashboard import build_dashboard_view_model, render_dashboard_html

WORKDIR = ROOT / "research" / EXP
RUNTIME = ROOT / "runtime" / "research" / EXP
if RUNTIME.exists():
    shutil.rmtree(RUNTIME)
RUNTIME.mkdir(parents=True, exist_ok=True)

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def base_dashboard() -> dict:
    return {
        "schema_version": "dashboard.model_matrix.v1",
        "artifact_id": "unit_dashboard",
        "top_models": [
            {
                "rank": 1,
                "model": "ridge_mse__all_features",
                "base_model": "ridge_mse",
                "feature_set": "all_features",
                "avg_position": 0.5,
                "return_pct": 2.0,
                "relative_return_pct": 0.5,
                "max_drawdown_pct": 4.0,
                "stability_score": 3.0,
            }
        ],
    }


In [2]:
cases = {}

strong = base_dashboard()
strong["manifest_source"] = {
    "kind": "analysis_manifest",
    "path": "matrix_card/analysis_manifest.json",
    "schema_version": "artifact.v1",
    "manifest_hash": "a" * 64,
    "producer": "src.analysis.exports",
}
cases["strong_analysis_manifest"] = strong

missing = base_dashboard()
cases["missing_manifest_source"] = missing

fixed = base_dashboard()
fixed["manifest_source"] = {
    "kind": "fixed_sidecars",
    "path": "",
    "manifest_hash": "",
    "producer": "src.dashboard.model_matrix",
}
cases["fixed_sidecars_manifest_source"] = fixed

results = []
html_by_case = {}
for case_name, payload in cases.items():
    view = build_dashboard_view_model(payload)
    html = render_dashboard_html(view, title=f"SOXL {case_name}")
    html_by_case[case_name] = html
    overview = view["readiness_overview"]
    gates = {gate["gate"]: gate for gate in overview["gates"]}
    manifest_gate = gates["dashboard_manifest"]
    results.append(
        {
            "case": case_name,
            "overview_status": overview["status"],
            "gate_status": manifest_gate["status"],
            "gate_detail": manifest_gate["detail"],
            "source_schema_version": manifest_gate["source_schema_version"],
            "html_contains_dashboard_manifest": "Dashboard Manifest" in html,
        }
    )

by_case = {row["case"]: row for row in results}
assert by_case["strong_analysis_manifest"]["overview_status"] == "ready"
assert by_case["strong_analysis_manifest"]["gate_status"] == "pass"
assert by_case["strong_analysis_manifest"]["gate_detail"].startswith("analysis_manifest / complete")
assert by_case["missing_manifest_source"]["overview_status"] == "blocked"
assert by_case["missing_manifest_source"]["gate_status"] == "block"
assert by_case["missing_manifest_source"]["gate_detail"] == "missing / missing hash "
assert by_case["fixed_sidecars_manifest_source"]["overview_status"] == "blocked"
assert by_case["fixed_sidecars_manifest_source"]["gate_status"] == "block"
assert by_case["fixed_sidecars_manifest_source"]["gate_detail"] == "fixed_sidecars / weak_kind hash "
assert all(row["html_contains_dashboard_manifest"] for row in results)

(WORKDIR / "dashboard_manifest_gate.html").write_text(html_by_case["strong_analysis_manifest"], encoding="utf-8")
summary = {
    "schema_version": "research.dashboard_manifest_readiness_gate_v1",
    "experiment": EXP,
    "ready_case_count": sum(1 for row in results if row["overview_status"] == "ready"),
    "blocked_case_count": sum(1 for row in results if row["overview_status"] == "blocked"),
    "cases": results,
}
write_json(WORKDIR / "analysis_summary.json", summary)
summary


{'schema_version': 'research.dashboard_manifest_readiness_gate_v1',
 'experiment': 'dashboard_manifest_readiness_gate_v1',
 'ready_case_count': 1,
 'blocked_case_count': 2,
 'cases': [{'case': 'strong_analysis_manifest',
   'overview_status': 'ready',
   'gate_status': 'pass',
   'gate_detail': 'analysis_manifest / complete hash aaaaaaaaaa',
   'source_schema_version': 'artifact.v1',
   'html_contains_dashboard_manifest': True},
  {'case': 'missing_manifest_source',
   'overview_status': 'blocked',
   'gate_status': 'block',
   'gate_detail': 'missing / missing hash ',
   'source_schema_version': '',
   'html_contains_dashboard_manifest': True},
  {'case': 'fixed_sidecars_manifest_source',
   'overview_status': 'blocked',
   'gate_status': 'block',
   'gate_detail': 'fixed_sidecars / weak_kind hash ',
   'source_schema_version': '',
   'html_contains_dashboard_manifest': True}]}

## 中文结论

- 做了什么：验证 dashboard view model 和 HTML 页面把 `manifest_source` 作为 `dashboard_manifest` readiness gate 展示。
- 核心指标：3 个场景中，`analysis_manifest` 且字段完整时 readiness=`ready`、gate=`pass`；缺失 manifest source 和 `fixed_sidecars` source 时 readiness=`blocked`、gate=`block`。
- regime 差异：本次是 dashboard/readiness 契约验证，不直接比较市场 regime；它保证后续 regime 分层、模型差异和 paper/API 状态能在同一个 review 页面上解释。
- 是否晋升：该能力进入 `src/dashboard` 作为统一 review 展示层；没有新增 production strategy。
- 风险：当前 gate 检查 dashboard manifest 来源字段，不递归校验 manifest 内每个 public artifact 的业务内容。
- 下一步：继续统一 production health、paper intent 和 dashboard view 的 manifest 判定 helper，减少不同层的规则漂移。